## Carregando Dados e Bibliotecas necessárias.

In [1]:
import pandas as pd
import numpy as np

data_layer_filepath = '../../data_layer/'

df = pd.read_csv(data_layer_filepath + 'raw/airbnb-dataset.csv', low_memory=False)
print("Dataset carregado com sucesso!")
df.head()

Dataset carregado com sucesso!


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


# Tratamento dos Dados.

## Padronização dos Nomes das Colunas.

In [2]:
df.rename(
    columns={col: col.lower().replace(' ', '_') for col in df.columns},
    inplace=True
)
print(df.columns)

Index(['id', 'name', 'host_id', 'host_identity_verified', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country',
       'country_code', 'instant_bookable', 'cancellation_policy', 'room_type',
       'construction_year', 'price', 'service_fee', 'minimum_nights',
       'number_of_reviews', 'last_review', 'reviews_per_month',
       'review_rate_number', 'calculated_host_listings_count',
       'availability_365', 'house_rules', 'license'],
      dtype='object')


## Remoção de Colunas Desnecessárias

Como quase todas as tuplas de **license** estavam como nulas, optamos por não trabalhar com essa coluna. Além disso, optamos por remover as colunas de **Country** e **Country_code** visto que sabemos que todas se enquadram no Estados Unidos e possuem o códido do país como "US".

In [3]:
cols_to_drop = ['country', 'country_code', 'license']
df.drop(columns=cols_to_drop, inplace=True)

for col in cols_to_drop:
    if col not in df.columns:
        print(f"Coluna {col} deletada!")

print("Colunas: ")
print(df.columns)

Coluna country deletada!
Coluna country_code deletada!
Coluna license deletada!
Colunas: 
Index(['id', 'name', 'host_id', 'host_identity_verified', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'lat', 'long',
       'instant_bookable', 'cancellation_policy', 'room_type',
       'construction_year', 'price', 'service_fee', 'minimum_nights',
       'number_of_reviews', 'last_review', 'reviews_per_month',
       'review_rate_number', 'calculated_host_listings_count',
       'availability_365', 'house_rules'],
      dtype='object')


## Correções dos Tipos de Dados.

1) **price** e **service_fee** possuem o caracter especial "$" e estão como object. Com isso, iremos altera-las para o tipo númerico (Float)

In [4]:
money_columns = ['price', 'service_fee']

print("Tipos de dados antes da correção:")
print(df[money_columns].dtypes)

for col in money_columns:
    df[col] = df[col].str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip().astype(float)

print("Tipos de dados corrigidos:")
print(df[money_columns].dtypes)

Tipos de dados antes da correção:
price          object
service_fee    object
dtype: object
Tipos de dados corrigidos:
price          float64
service_fee    float64
dtype: object


2) **construction_year**, **availability_365** e **calculated_host_listings_count**: Não faria sentido estar sendo guardado em Float já que era o ano de construção. Ou seja, nunca viria um número decimal. O mesmo vale para availability_365 e para calculated_host_listings_count.

In [5]:
floats_to_ints = ['availability_365', 'construction_year','calculated_host_listings_count']

for col in floats_to_ints:
    print(f"Tipo de dado da chave {col} antes da correção: {df[col].dtype}")

    df.dropna(subset=[col], inplace=True)
    df[col] = df[col].astype('int64')

    print(f"Tipo de dado da chave {col} após a correção: {df[col].dtype}")

Tipo de dado da chave availability_365 antes da correção: float64
Tipo de dado da chave availability_365 após a correção: int64
Tipo de dado da chave construction_year antes da correção: float64
Tipo de dado da chave construction_year após a correção: int64
Tipo de dado da chave calculated_host_listings_count antes da correção: float64
Tipo de dado da chave calculated_host_listings_count após a correção: int64


3) **host_identity_verified**: A coluna host_identity_verified pode ser transformada em booleano (True/False), o que é mais eficiente e semanticamente correto.

In [6]:
df['host_identity_verified'] = df['host_identity_verified'].astype(str).str.lower().str.strip()

to_replace = {
    'verified': True,
    'unconfirmed': False
}

df['host_identity_verified'] = df['host_identity_verified'].replace(to_replace).astype(bool)


print("Tipo de dado da coluna APÓS o tratamento:")
print(df['host_identity_verified'].dtype)
print("\nValores únicos na coluna APÓS o tratamento:")
print(df['host_identity_verified'].unique())
print("\nContagem de valores na coluna APÓS o tratamento:")
print(df['host_identity_verified'].value_counts(dropna=False))

Tipo de dado da coluna APÓS o tratamento:
bool

Valores únicos na coluna APÓS o tratamento:
[False  True]

Contagem de valores na coluna APÓS o tratamento:
host_identity_verified
True     50916
False    50724
Name: count, dtype: int64


4. **Colunas object**: algumas colunas tem o tipo misto `object`. Na análise realizada na camada bronze, concluímos que essas colunas devem ter o tipo `string`. Além disso, a coluna `instant_bookable` deveria ter o tipo `bool`. Por fim, a coluna `last_review` deveria ter um tipo de dados que melhor representa uma data.

In [7]:
string_cols = [
    'name',
    'host_name',
    'neighbourhood_group',
    'neighbourhood',
    'cancellation_policy',
    'room_type',
    'house_rules',
]

for col in string_cols:
    df[col] = df[col].astype('string')

df['instant_bookable'] = df['instant_bookable'].astype(bool)

df['last_review'] = pd.to_datetime(df['last_review'])

5. **minimum_nights**: esta coluna é do tipo `float64`, mas deveria ter um tipo inteiro, visto que trata da quantidade de noites mínimas que um interessado deve passar num lugar anunciado.

In [8]:
df.dropna(subset=['minimum_nights'], inplace=True)
df['minimum_nights'] = df['minimum_nights'].astype('int64')

## Correção de Inconsistência nos Dados.

### Correção nos erros de digitação no nome dos bairros que apresentavam "brookln" e "manhatan"

In [9]:

df['neighbourhood_group'] = df['neighbourhood_group'].replace({
    'brookln': 'Brooklyn',
    'manhatan': 'Manhattan'
})


print(df['neighbourhood_group'].unique())

<StringArray>
['Brooklyn', 'Manhattan', <NA>, 'Queens', 'Staten Island', 'Bronx']
Length: 6, dtype: string


### Tratamento de Valores Ausentes.


1) Remoção de Anúncios sem preço.

In [10]:

df.dropna(subset=['price', 'service_fee'], inplace=True)

print(f"Valores nulos em 'price' após remoção: {df['price'].isnull().sum()}")

Valores nulos em 'price' após remoção: 0


2) Criação de Coluna booleana para house_rules: Como metade dos valores é nulo, iremos criar uma nova coluna para indicar se essa "casa" possui ou não regras definidas.

In [11]:

df['has_house_rules'] = df['house_rules'].notna()

# Podemos agora remover a coluna original se o conteúdo de texto não for usado
# df_silver.drop(columns=['house_rules'], inplace=True)


print(df['has_house_rules'].value_counts())

has_house_rules
False    51240
True     49544
Name: count, dtype: int64


3) Preenchimento dos poucos anúncios sem nome (ou sem nome de host) com "Sem nome informado"

In [12]:
for col in ['name', 'host_name']:
    df[col] = df[col].fillna('Sem nome informado')

4) Remoção dos demais **nans**

In [13]:
nans_to_drop = [
    'neighbourhood', 
    'neighbourhood_group', 
    'lat', 
    'long',
    'host_identity_verified',
    'minimum_nights',
]

df.dropna(subset=nans_to_drop, inplace=True)

## Tratamentos de Valores invalidados.

Foi identificado alguns valores negativos na coluna **availability_365** que não fazem sentido com o escopo.

In [14]:
df = df[df['availability_365'] >= 0]
print(df['availability_365'].min())

0


## Dicionario - Silver

In [15]:
print("--- Amostra Aleatória de 20 Linhas do DataFrame 'df_silver' ---")

# O .sample(20) pega 20 linhas aleatórias do DataFrame.
# O .reset_index(drop=True) é para a visualização ficar mais limpa, sem o índice antigo.
display(df.sample(20).reset_index(drop=True))


print("\n\n--- Resumo das Informações (Info) ---")
df.info()

--- Amostra Aleatória de 20 Linhas do DataFrame 'df_silver' ---


,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,instant_bookable,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,has_house_rules
0,31367342,"Simple, Cozy Private Room Near Beach & Subway",97675668504,False,Anna,Queens,Arverne,40.59154,-73.79116,False,...,153.0,1,45.0,2022-02-09,0.71,5.0,4,169,<NA>,False
1,53416815,VERY BIG APARTMENT IN HARLEM/ENTIRE APARTMENT,56613905397,True,Maxime C And Winie C,Manhattan,Harlem,40.81634,-73.94344,False,...,152.0,2,3.0,2018-10-26,0.24,2.0,3,175,Please expect to share the apartment with gues...,True
2,49064691,Luxury For Less In Downtown Manhattan,58757393793,False,Lauren,Manhattan,Battery Park City,40.71617,-74.01468,False,...,62.0,1,1.0,2016-11-05,0.03,1.0,1,0,<NA>,False
3,22984533,Simple studio,93608331418,True,Xochilth,Manhattan,Harlem,40.82211,-73.93887,False,...,127.0,4,0.0,NaT,NaN,3.0,1,3,<NA>,False
4,20170564,"Stylish 1 bedroom, Upper East Side, E91st",27968310617,True,Mariah,Manhattan,Upper East Side,40.78051,-73.94809,False,...,107.0,1,17.0,2019-06-23,1.62,3.0,1,5,Please remember that this is a residential bui...,True
5,18854434,Wyndham Midtown 45 (2 Bedroom Presidential) 3A,21775207158,False,Michael,Manhattan,Midtown,40.75305,-73.97129,False,...,168.0,3,14.0,2018-11-19,1.13,5.0,12,0,"Sorry, no smoking. You'll be sharing the house...",True
6,42467468,Private Apt (2 Bedrooms includes 1 Bath & Kitc...,57211221438,True,Brian,Manhattan,Two Bridges,40.71253,-73.99599,True,...,38.0,3,19.0,2019-07-02,5.43,5.0,1,42,<NA>,False
7,37743094,Large 1BR Apartment in Best LES Location,80090663770,False,Kyra,Manhattan,Lower East Side,40.71875,-73.98638,False,...,161.0,30,1.0,2020-11-05,0.06,5.0,3,365,<NA>,False
8,56524607,Luxurious LES/Nolita Loft,28592162754,True,Nise,Manhattan,Lower East Side,40.72018,-73.98521,True,...,239.0,3,53.0,2019-06-06,1.98,4.0,1,220,"No drugs, smoking, or guests. Keep the tv volu...",True
9,31462338,Harlem 420 Marijuana Consumption Room @ Castle...,15841617563,True,Sidney,Manhattan,Harlem,40.80858,-73.95550,False,...,35.0,2,5.0,2022-02-07,0.19,3.0,3,89,<NA>,False




--- Resumo das Informações (Info) ---
<class 'pandas.core.frame.DataFrame'>
Index: 100334 entries, 0 to 102598
Data columns (total 24 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   id                              100334 non-null  int64         
 1   name                            100334 non-null  string        
 2   host_id                         100334 non-null  int64         
 3   host_identity_verified          100334 non-null  bool          
 4   host_name                       100334 non-null  string        
 5   neighbourhood_group             100334 non-null  string        
 6   neighbourhood                   100334 non-null  string        
 7   lat                             100334 non-null  float64       
 8   long                            100334 non-null  float64       
 9   instant_bookable                100334 non-null  bool          
 10  cancellation_policy  

## Salvando Dataset

In [16]:

df.to_csv(data_layer_filepath + 'silver/airbnb-dataset-silver.csv', index=False)

print("Dataset da camada Silver salvo com sucesso!")

Dataset da camada Silver salvo com sucesso!


In [19]:
df['construction_year'].isnull().sum()

np.int64(0)